Simulator for the independent Euclid-VIS-like lens test set (lenstronomy).
All classes (compound, single, nearby, non_lens) share ONE galaxy+gate sampler (draw_system).


In [1]:
import warnings
import numpy as np
from astropy.cosmology import FlatLambdaCDM
from lenstronomy.SimulationAPI.sim_api import SimAPI
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.Util import param_util
 
warnings.filterwarnings("ignore")

In [2]:
#Setting parameters
NUMPIX = 100
BAND = {
    'pixel_scale': 0.1, 'seeing': 0.16, 'exposure_time': 565, 'num_exposures': 4,
    'magnitude_zero_point': 25.6, 'read_noise': 4.5, 'ccd_gain': 3.1, 'sky_brightness': 22.4,
    'psf_type': 'GAUSSIAN',
}
KWARGS_NUMERICS = {'supersampling_factor': 3}
ZP = BAND['magnitude_zero_point']
T_TOT = BAND['exposure_time'] * BAND['num_exposures']      # total exposure [s]
COSMO = FlatLambdaCDM(H0=70, Om0=0.3)
C_KMS = 299792.458
ARCSEC_PER_RAD = 206264.806

In [3]:
#Parameter specs derived from existing confirmed 4 lenses + researched values
RNG_LENS_MAG = (20.0, 22.3)
RNG_S1_MAG = (22.3, 23.3)
RNG_S2_MAG = (23.2, 24.4)
RNG_BETA = (0.5, 0.85)
RNG_THE1 = (0.45, 1.2)
RNG_THE2 = (0.9, 1.5)
RNG_Q = (0.6, 0.8)
RNG_GAMMA = (1.6, 2.2)
RNG_SIS1 = (0.1, 0.5)      # chosen to bracket the 4 real systems (0.13-0.48")
MIN_RING_SEP = 0.15        # theta_E2-theta_E1 [arcsec]; real systems: 0.17-0.86"
NMIN_VIS = 20              # pixels above 3 sigma (local noise incl. lens-light Poisson) required per arc
NSIG_VIS = 3.0
MU_RANGE = (2.0, 60.0)     # total magnification allowed for a source
 

In [4]:
#cosmology tables (flat -> analytic D_ls)
_zg = np.linspace(0, 8, 8001)
_chi = COSMO.comoving_distance(_zg).value          # Mpc
def chi(z): return np.interp(z, _zg, _chi)
def z_of_chi(c): return np.interp(c, _chi, _zg)
def f_ds(zl, zs): return 1.0 - chi(zl) / chi(zs)               # D_ls/D_s (flat universe)
def arcsec_per_kpc(z):
    da = chi(z) / (1 + z) * 1e3                                 # kpc
    return ARCSEC_PER_RAD / da
def theta_inf(sigma): return 4 * np.pi * (sigma / C_KMS) ** 2 * ARCSEC_PER_RAD   # SIS, f_ds=1
def zs_from_f(zl, f): return float(z_of_chi(chi(zl) / (1.0 - f)))
 

In [5]:
#lens-galaxy population (Faber-Jackson)
ZL_RANGE = (0.3, 0.7)
FJ_SCATTER = 0.25
R0_KPC = 5.0                # R_e at sigma=260 km/s; R_e ~ sigma^1.5
def _fj_raw(sigma, zl, eps):
    return -10 * np.log10(sigma / 260.0) + COSMO.distmod(zl).value + eps
def _calibrate_FJ():
    r = np.random.default_rng(12345); n = 200000
    s = r.normal(260, 40, 2 * n); s = s[(s > 170) & (s < 380)][:n]
    z = r.uniform(*ZL_RANGE, len(s))
    raw = _fj_raw(s, z, r.normal(0, FJ_SCATTER, len(s)))
    return 0.5 * sum(RNG_LENS_MAG) - np.median(raw)           # centre the distribution on the target window
FJ_C = _calibrate_FJ()
 
 
def draw_galaxy(rng):
    while True:
        sigma = rng.normal(260, 40)
        if 170 < sigma < 380:
            break
    zl = rng.uniform(*ZL_RANGE)
    mag = FJ_C + float(_fj_raw(sigma, zl, rng.normal(0, FJ_SCATTER)))
    R = R0_KPC * (sigma / 260.0) ** 1.5 * 10 ** rng.normal(0, 0.08) * arcsec_per_kpc(zl)
    R = float(np.clip(R, 0.3, 2.0))
    return dict(zl=zl, sigma=sigma, mag=mag, R=R, n=rng.uniform(3, 5), q=rng.uniform(0.6, 0.95),
                phi=rng.uniform(-np.pi / 2, np.pi / 2))
 
 
def draw_system(rng):
    """Shared by ALL classes (non-lens included): galaxy + lensing geometry + gates. Returns dict."""
    while True:
        th2 = rng.uniform(*RNG_THE2)
        g = draw_galaxy(rng)
        if not RNG_LENS_MAG[0] <= g['mag'] <= RNG_LENS_MAG[1]:
            continue
        zl = g['zl']; tinf = theta_inf(g['sigma'])
        f2 = th2 / tinf                                              # required D_ls/D_s of the outer/only source
        if not (f_ds(zl, zl + 0.6) <= f2 <= f_ds(zl, 4.0)):
            continue                                                 # no physical source redshift gives this theta_E
        zs2 = zs_from_f(zl, f2)
        beta = rng.uniform(*RNG_BETA); gam = rng.uniform(*RNG_GAMMA)
        th1 = th2 * beta ** (1.0 / (gam - 1.0))                      # EPL scaling of the deflection by beta
        if not (RNG_THE1[0] <= th1 <= RNG_THE1[1] and th2 - th1 >= MIN_RING_SEP):
            continue
        zs1 = zs_from_f(zl, beta * f2)                               # D_ls1/D_s1 = beta * D_ls2/D_s2
        if zs1 < zl + 0.15 or zs2 - zs1 < 0.2:
            continue
        qm = rng.uniform(*RNG_Q)
        return dict(g=g, th1=th1, th2=th2, beta=beta, gam=gam, zs1=zs1, zs2=zs2,
                    th_sis1=rng.uniform(*RNG_SIS1), q_m=qm, phi_m=g['phi'] + rng.normal(0, np.deg2rad(8)),
                    shear=rng.uniform(0.01, 0.08), phi_sh=rng.uniform(0, np.pi))
 

In [6]:
#light profiles, field, neighbour
def sersic_kw(mag, R, n, q, phi, x, y):
    e1, e2 = param_util.phi_q2_ellipticity(phi, q)
    return {'magnitude': mag, 'R_sersic': R, 'n_sersic': n, 'e1': e1, 'e2': e2, 'center_x': x, 'center_y': y}
 
 
def sample_field(rng, min_r=2.5, n_max=2):
    """Same rule for every class (notebook used a lens-dependent min_r -> leak)."""
    out = []
    for _ in range(int(rng.integers(0, n_max + 1))):
        while True:
            x, y = rng.uniform(-4.5, 4.5, 2)
            if np.hypot(x, y) > min_r:
                break
        out.append(sersic_kw(rng.uniform(22.5, 25.5), rng.uniform(0.15, 0.5), rng.uniform(1, 2),
                             rng.uniform(0.3, 1), rng.uniform(0, np.pi), x, y))
    return out
 
 
def sample_source(rng, z, th_ref):
    r, a = th_ref * rng.uniform(0.02, 0.4), rng.uniform(0, 2 * np.pi)
    return dict(x=r * np.cos(a), y=r * np.sin(a), R=0.3 / (1 + 0.5 * z) * rng.uniform(0.7, 1.3),
                n=rng.uniform(0.8, 2.0), q=rng.uniform(0.4, 1.0), phi=rng.uniform(0, np.pi), z=z)
 

In [7]:
#rendering machinery
_SIMS = {}
def _sim():
    if 'sim' not in _SIMS:
        _SIMS['sim'] = SimAPI(NUMPIX, BAND, {'lens_model_list': ['SIE'], 'lens_light_model_list': ['SERSIC_ELLIPSE'],
                                             'source_light_model_list': ['SERSIC_ELLIPSE']})
        s = _SIMS['sim']
        _SIMS['sigma_bkg'] = float(s.noise_for_model(model=np.zeros((NUMPIX, NUMPIX)), background_noise=True,
                                                     poisson_noise=False).std())
    return _SIMS['sim']
 
def sigma_bkg():
    _sim(); return _SIMS['sigma_bkg']
 
def _amp(kw, kind):
    s = _sim()
    if kind == 'lens_light':
        return s.magnitude2amplitude(kwargs_lens_light_mag=[kw])[0][0]
    return s.magnitude2amplitude(kwargs_source_mag=[kw])[1][0]
 
def render_unlensed(kws):
    """Sum of unlensed Sersic components (lens light, field, neighbour); noiseless, counts/s."""
    if not kws:
        return np.zeros((NUMPIX, NUMPIX))
    s = _sim()
    amps = [_amp(k, 'lens_light') for k in kws]
    im = ImageModel(s.data_class, s.psf_class, lens_light_model_class=LightModel(['SERSIC_ELLIPSE'] * len(kws)),
                    kwargs_numerics=KWARGS_NUMERICS)
    return im.image(kwargs_lens_light=amps)
 
REF_MAG = 25.0
def render_lensed_ref(lens_model, kwargs_lens, src):
    """Lensed image of one source at reference intrinsic mag; returns image, lensed mag, magnification."""
    s = _sim()
    kw = sersic_kw(REF_MAG, src['R'], src['n'], src['q'], src['phi'], src['x'], src['y'])
    amp = _amp(kw, 'source')
    im = ImageModel(s.data_class, s.psf_class, lens_model_class=lens_model,
                    source_model_class=LightModel(['SERSIC_ELLIPSE']), kwargs_numerics=KWARGS_NUMERICS)
    img = im.image(kwargs_lens=kwargs_lens, kwargs_source=[amp])
    F = img.sum()
    if F <= 0:
        return img, np.inf, 0.0
    m_l = ZP - 2.5 * np.log10(F)
    mu = F / 10 ** (-0.4 * (REF_MAG - ZP))
    return img, m_l, mu
 
def scale_to_lensed_mag(img, m_l_ref, m_target):
    """Flux is linear in amplitude: scale so the lensed total equals m_target. Returns arc, intrinsic mag."""
    d = m_target - m_l_ref
    return img * 10 ** (-0.4 * d), REF_MAG + d
 
def n_vis(arc, other):
    var = sigma_bkg() ** 2 + np.clip(arc + other, 0, None) / T_TOT
    return int((arc > NSIG_VIS * np.sqrt(var)).sum())
 
def add_noise(img):
    return (img + _sim().noise_for_model(model=img, background_noise=True, poisson_noise=True)).astype(np.float32)
 

In [8]:
# lens models
def _epl_shear(th, gam, q, phi, sh, phi_sh):
    e1, e2 = param_util.phi_q2_ellipticity(phi, q)
    return [{'theta_E': th, 'gamma': gam, 'e1': e1, 'e2': e2, 'center_x': 0.0, 'center_y': 0.0},
            {'gamma1': sh * np.cos(2 * phi_sh), 'gamma2': sh * np.sin(2 * phi_sh), 'ra_0': 0.0, 'dec_0': 0.0}]
 
_LM = {}
def lm_single():
    if 'a' not in _LM: _LM['a'] = LensModel(['EPL', 'SHEAR'])
    return _LM['a']
def lm_multi(zl, zs1, zs2):
    return LensModel(['EPL', 'SHEAR', 'SIS'], multi_plane=True, lens_redshift_list=[zl, zl, zs1],
                     z_source=zs2, cosmo=COSMO)
 
 

In [9]:
#class makers
def _target_mag(rng, which):
    return rng.uniform(*{'s1': RNG_S1_MAG, 's2': RNG_S2_MAG}[which])
 
def _common_light(sysd, rng):
    g = sysd['g']
    gal = sersic_kw(g['mag'], g['R'], g['n'], g['q'], g['phi'], 0.0, 0.0)
    field = sample_field(rng)
    return gal, field, render_unlensed([gal] + field)
 
def _base_rec(sysd, cls):
    g = sysd['g']
    return dict(class_name=cls, z_l=g['zl'], sigma_v=g['sigma'], lens_mag=g['mag'], lens_R=g['R'], lens_n=g['n'],
                lens_q_light=g['q'], lens_phi_light=g['phi'])
 
def make_non_lens(rng):
    sysd = draw_system(rng)
    gal, field, base = _common_light(sysd, rng)
    rec = _base_rec(sysd, 'non_lens')
    rec.update(n_field=len(field))
    return add_noise(base), rec
 
def _single_arc(rng, sysd, base, extra_light=None, tries=40):
    th, gam = sysd['th2'], sysd['gam']
    kl = _epl_shear(th, gam, sysd['q_m'], sysd['phi_m'], sysd['shear'], sysd['phi_sh'])
    for _ in range(tries):
        src = sample_source(rng, sysd['zs2'], th)
        img, m_l, mu = render_lensed_ref(lm_single(), kl, src)
        if not MU_RANGE[0] <= mu <= MU_RANGE[1]:
            continue
        which = 's1' if rng.random() < 0.5 else 's2'
        arc, m_int = scale_to_lensed_mag(img, m_l, _target_mag(rng, which))
        other = base if extra_light is None else base + extra_light
        nv = n_vis(arc, other)
        if nv >= NMIN_VIS:
            return arc, dict(src=src, m_int=m_int, m_lensed=ZP - 2.5 * np.log10(arc.sum()), mu=mu, nvis=nv,
                             pool=which)
    return None
 
def make_single(rng):
    while True:
        sysd = draw_system(rng)
        gal, field, base = _common_light(sysd, rng)
        res = _single_arc(rng, sysd, base)
        if res is None:
            continue
        arc, d = res
        rec = _base_rec(sysd, 'single')
        rec.update(theta_E=sysd['th2'], theta_E_outer=sysd['th2'], gamma=sysd['gam'], q_mass=sysd['q_m'],
                   phi_mass=sysd['phi_m'], shear=sysd['shear'], z_s=sysd['zs2'], s_mag_lensed=d['m_lensed'],
                   s_mag_int=d['m_int'], s_mu=d['mu'], s_nvis=d['nvis'], s_pool=d['pool'], n_field=len(field))
        return add_noise(base + arc), rec
 
def make_nearby(rng):
    while True:
        sysd = draw_system(rng)
        gal, field, base = _common_light(sysd, rng)
        th = sysd['th2']
        ang, d = rng.uniform(0, 2 * np.pi), th * rng.uniform(1.3, 2.0) + 0.3      # outside the ring (notebook rule)
        nb = sersic_kw(rng.uniform(22.5, 24.5), rng.uniform(0.15, 0.35), 1.0, rng.uniform(0.2, 0.45),
                       ang + np.pi / 2 + rng.normal(0, 0.25), d * np.cos(ang), d * np.sin(ang))   # long axis tangential
        nb_img = render_unlensed([nb])
        if n_vis(nb_img, base) < NMIN_VIS:                                          # same visibility rule
            continue
        res = _single_arc(rng, sysd, base, extra_light=nb_img)
        if res is None:
            continue
        arc, dd = res
        rec = _base_rec(sysd, 'nearby')
        rec.update(theta_E=th, theta_E_outer=th, gamma=sysd['gam'], q_mass=sysd['q_m'], phi_mass=sysd['phi_m'],
                   shear=sysd['shear'], z_s=sysd['zs2'], s_mag_lensed=dd['m_lensed'], s_mag_int=dd['m_int'],
                   s_mu=dd['mu'], s_nvis=dd['nvis'], s_pool=dd['pool'], n_field=len(field),
                   nb_mag=nb['magnitude'], nb_R=nb['R_sersic'], nb_dist=d, nb_n_vis=n_vis(nb_img, base))
        return add_noise(base + arc + nb_img), rec
 
def make_compound(rng):
    while True:
        sysd = draw_system(rng)
        gal, field, base = _common_light(sysd, rng)
        zl, zs1, zs2, beta = sysd['g']['zl'], sysd['zs1'], sysd['zs2'], sysd['beta']
        gam, th1, th2 = sysd['gam'], sysd['th1'], sysd['th2']
        # source 1: single plane, EPL scaled by D_ls1/D_s1 ; shear scales linearly
        kl1 = _epl_shear(th1, gam, sysd['q_m'], sysd['phi_m'], beta * sysd['shear'], sysd['phi_sh'])
        # source 2: multi-plane [EPL, SHEAR @ z_l ; SIS @ z_s1], all parameters defined for z_source = z_s2
        for _ in range(40):
            s1 = sample_source(rng, zs1, th1); s2 = sample_source(rng, zs2, th2)
            img1, ml1, mu1 = render_lensed_ref(lm_single(), kl1, s1)
            if not MU_RANGE[0] <= mu1 <= MU_RANGE[1]:
                continue
            kl2 = _epl_shear(th2, gam, sysd['q_m'], sysd['phi_m'], sysd['shear'], sysd['phi_sh']) + \
                [{'theta_E': sysd['th_sis1'], 'center_x': s1['x'], 'center_y': s1['y']}]   # mass of source 1
            img2, ml2, mu2 = render_lensed_ref(lm_multi(zl, zs1, zs2), kl2, s2)
            if not MU_RANGE[0] <= mu2 <= MU_RANGE[1]:
                continue
            a1, mi1 = scale_to_lensed_mag(img1, ml1, _target_mag(rng, 's1'))
            a2, mi2 = scale_to_lensed_mag(img2, ml2, _target_mag(rng, 's2'))
            nv1, nv2 = n_vis(a1, base + a2), n_vis(a2, base + a1)
            if nv1 < NMIN_VIS or nv2 < NMIN_VIS:
                continue
            rec = _base_rec(sysd, 'compound')
            rec.update(theta_E1=th1, theta_E2=th2, theta_E_outer=th2, theta_sis1=sysd['th_sis1'], beta=beta,
                       gamma=gam, q_mass=sysd['q_m'], phi_mass=sysd['phi_m'], shear=sysd['shear'], z_s1=zs1,
                       z_s2=zs2, s1_mag_lensed=ZP - 2.5 * np.log10(a1.sum()), s2_mag_lensed=ZP - 2.5 * np.log10(a2.sum()),
                       s1_mag_int=mi1, s2_mag_int=mi2, s1_mu=mu1, s2_mu=mu2, s1_nvis=nv1, s2_nvis=nv2,
                       n_field=len(field))
            return add_noise(base + a1 + a2), rec
 
MAKERS = {'compound': make_compound, 'single': make_single, 'non_lens': make_non_lens, 'nearby': make_nearby}
 


Generate the test set 1000/1000/500/500


In [10]:
import os, sys, numpy as np, pandas as pd
from multiprocessing import Pool

In [11]:
MASTER_SEED = 20261104
COUNTS = {'compound': 1000, 'single': 1000, 'non_lens': 500, 'nearby': 500}
OUT = 'data/cutouts/test'
 
 
def job(args):
    slot, cls, ss = args
    rng = np.random.default_rng(ss)
    np.random.seed(int(ss.generate_state(1)[0]))          # lenstronomy noise uses the global numpy RNG
    img, rec = S.MAKERS[cls](rng)
    rec['slot'] = slot
    return slot, img, rec

In [13]:
if __name__ == '__main__':
    os.makedirs(OUT, exist_ok=True)
    classes = [c for c, n in COUNTS.items() for _ in range(n)]
    N = len(classes)
    root = np.random.SeedSequence(MASTER_SEED)
    children = root.spawn(N + 1)
    perm = np.random.default_rng(children[-1]).permutation(N)        # opaque ids: class is NOT grouped by id
    jobs = [(i, classes[i], children[i]) for i in range(N)]
    recs = []
    

In [ ]:
with Pool(2) as p:
        for k, (slot, img, rec) in enumerate(p.imap_unordered(job, jobs, chunksize=8)):
            sid = f'sim_{perm[slot]:07d}'
            assert img.dtype == np.float32 and img.shape == (100, 100) and np.isfinite(img).all()
            np.save(f'{OUT}/{sid}.npy', img)
            rec['id'] = sid
            rec['seed_entropy'] = MASTER_SEED; rec['spawn_index'] = slot
            recs.append(rec)
            if k % 250 == 0:
                print(k, '/', N, flush=True)
        lab = pd.DataFrame(recs).sort_values('id').reset_index(drop=True)
        lab['is_compound'] = (lab.class_name == 'compound').astype(int)
        lead = ['id', 'class_name', 'is_compound']
        lab = lab[lead + [c for c in lab.columns if c not in lead]]
        lab.to_csv('labels.csv', index=False)
 
        #same columns/format as the 79 image dataset in github 
        r = np.random.default_rng(MASTER_SEED + 1)
        ra = np.round(r.uniform(0, 360, N), 6)
        dec = np.round(np.degrees(np.arcsin(r.uniform(-1, 1, N))), 6)       # unrelated to class, shuffled by id
        man = pd.DataFrame({'id': lab.id, 'label': (lab.class_name != 'non_lens').astype(int), 'grade': 'A', 'notes': '',
                            'ra': ra, 'dec': dec, 'path': [f'{OUT}/{i}.npy' for i in lab.id], 'split': 'test'})
        assert man[['ra', 'dec']].duplicated().sum() == 0 and man.id.is_unique
        man.to_csv('sim_data.csv', index=False)
        print(lab.class_name.value_counts())
 

Process SpawnPoolWorker-379:
Process SpawnPoolWorker-378:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/process.py", line 315, in _bootstrap
    self.run()
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/process.py", line 108, in run
    self

KeyboardInterrupt: 